# m43 lab · Foundation models, retrieval and evaluating language models

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You pretrain a small ECG-beat encoder without labels using a contrastive loss, then measure what it buys with
few labels; see how the pretraining cohort shapes what the encoder learns; check why attention divides by
$\sqrt d$; build a retrieval step over this course's own glossary; and build an evaluation harness for
language-model answers that runs offline, with a scripted stand-in model.

| Part | What you do | Data |
| --- | --- | --- |
| A | The contrastive (InfoNCE) loss and its gradient by hand (eq-m43-infonce, eq-m43-infonce-grad) | numbers |
| B | Pretrain without labels, then a linear probe with 10 to 300 labels | synthetic beats |
| C | The pretraining cohort matters: an encoder that never saw a wide QRS | synthetic beats |
| D | Attention and the square root of d (eq-m43-attention, eq-m43-scaling) | random numbers |
| E | Retrieval over the course glossary and formula sheet (eq-m43-tfidf, eq-m43-cosine) | course text |
| F | An evaluation harness: accuracy with its interval, consistency, citations (eq-m43-wilson, eq-m43-questions) | scripted stand-in |
| G | Real ECGs: pretraining on unlabeled PTB-XL beats, then a 10-record probe | PTB-XL (download) |
| H | Optional: the same contrastive pretraining with a 1-D CNN in PyTorch | synthetic beats |

The synthetic beat is the Gaussian-wave beat of m37 with patient-level wave sizes and QRS width; all its ranges
are illustrative. Nothing in this lab calls an external AI service. Each part ends with **checks**; the lab runs
in about a minute (plus the download in part G).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import foundation as fm
from epml import retrieval as rt
from epml.ml import auc_rank

THRESHOLD = 1.4   # QRS width factor above which a synthetic patient is labelled "wide" (illustrative)

## Part A · The contrastive loss by hand

For a batch of $N$ anchors with similarities $S_{ik}$ to $N$ candidates, where candidate $i$ is anchor $i$'s
positive, eq-m43-infonce is a cross-entropy over candidates. When every similarity is equal the positive is one
of $N$ equally likely candidates, so the loss is $\ln N$: that is the chance level to beat.

In [ ]:
N, tau = 128, 0.1
print(f"Equal similarities: loss {fm.infonce(np.zeros((N, N)), tau):.4f}, ln N = {np.log(N):.4f}")

rng = np.random.default_rng(0)
S = rng.uniform(-1, 1, (6, 6))
G = fm.infonce_grad(S, 0.2)
eps = 1e-6
fd = np.array([[(fm.infonce(S + eps * np.eye(6)[i][:, None] * np.eye(6)[k], 0.2)
                 - fm.infonce(S - eps * np.eye(6)[i][:, None] * np.eye(6)[k], 0.2)) / (2 * eps)
                for k in range(6)] for i in range(6)])
print(f"Largest difference, analytic vs finite-difference gradient: {np.abs(G - fd).max():.1e}")

# the temperature sets how sharply the loss rewards a margin between positive and negatives
margin = np.linspace(0, 1, 6)
for t in (0.05, 0.1, 0.5):
    losses = [fm.infonce(np.eye(8) * m, t) for m in margin]
    print(f"tau = {t:4.2f}: loss for a positive-minus-negative margin of 0, 0.2, ..., 1:",
          " ".join(f"{l:5.2f}" for l in losses))

In [ ]:
# Checks
assert abs(fm.infonce(np.zeros((N, N)), tau) - np.log(N)) < 1e-12
assert np.abs(G - fd).max() < 1e-7
assert fm.infonce(np.eye(8), 0.05) < fm.infonce(np.eye(8), 0.5)

## Part B · Pretrain without labels, then probe with few labels

Pretraining: 2,000 synthetic patients, no labels. Each step takes 128 patients and two different recordings of
each; a recording differs from the next by small changes in the beat, a time shift, baseline wander and noise.
The encoder is a one-hidden-layer network; its 64 hidden units are the features, and a 16-dimensional projection
feeds the loss (as in SimCLR, Chen et al. 2020).

Downstream task: is the QRS wide (width factor above 1.4)? We train a logistic regression on $n$ labelled patients
and test on 1,000 others, with three inputs: the raw samples, the features of an untrained (random) encoder, and
the features of the pretrained encoder. Each point is the mean of five random labelled sets.

In [ ]:
rng = np.random.default_rng(1)
pre_params = fm.patient_params(2000, rng)
std = fm.Standardizer(fm.recordings(pre_params, np.arange(2000), rng))
enc, losses = fm.pretrain(pre_params, std, steps=400, seed=2)
rand_enc = fm.Encoder(fm.BEAT_T.size, seed=9)
print(f"Loss: first 20 steps {losses[:20].mean():.2f}, last 20 steps {losses[-20:].mean():.2f}, chance {np.log(128):.2f}")

test = fm.patient_params(1000, np.random.default_rng(5))
Xt = std(fm.recordings(test, np.arange(1000), np.random.default_rng(6)))

def probe_auc(features, n, task="width", draws=5, lam=1.0):
    yt = (test["width"] > THRESHOLD) if task == "width" else (test["tf"] < 0.6)
    out = []
    for d in range(draws):
        r = np.random.default_rng(100 + d)
        while True:
            p = fm.patient_params(n, r)
            y = ((p["width"] > THRESHOLD) if task == "width" else (p["tf"] < 0.6)).astype(float)
            if 0 < y.sum() < n:
                break
        Xl = std(fm.recordings(p, np.arange(n), r))
        w = fm.fit_logistic(features(Xl), y, lam=lam)
        out.append(auc_rank(fm.logistic_score(w, features(Xt)), yt.astype(float)))
    return float(np.mean(out))

budgets = [10, 30, 100, 300]
curves = {"raw samples": [], "random encoder": [], "pretrained encoder": []}
for n in budgets:
    curves["raw samples"].append(probe_auc(lambda X: X, n, lam=10.0))
    curves["random encoder"].append(probe_auc(rand_enc.features, n))
    curves["pretrained encoder"].append(probe_auc(enc.features, n))
lowT = {name: probe_auc(f, 10, task="tf", lam=lam) for name, f, lam in
        [("raw samples", lambda X: X, 10.0), ("pretrained encoder", enc.features, 1.0)]}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
a1.plot(losses, lw=0.8); a1.axhline(np.log(128), ls="--", c="gray", label="chance, ln 128")
a1.set(xlabel="training step", ylabel="contrastive loss (nats)", title="Pretraining"); a1.legend()
for name, c in curves.items():
    a2.plot(budgets, c, "o-", label=name)
a2.set(xscale="log", xlabel="labelled patients", ylabel="test AUC (wide QRS)", title="Linear probe"); a2.legend()
plt.tight_layout(); plt.show()
for name, c in curves.items():
    print(f"{name:20s}", "  ".join(f"n={n}: {a:.3f}" for n, a in zip(budgets, c)))
print("Low T wave, 10 labels:", {k: round(v, 3) for k, v in lowT.items()})

In [ ]:
# Checks
assert losses[-20:].mean() < 0.6 * np.log(128)
gap = [p - r for p, r in zip(curves["pretrained encoder"], curves["raw samples"])]
assert gap[0] > 0.1                       # with 10 labels, pretraining helps a lot
assert gap[-1] < gap[0]                   # with 300 labels, much less
assert curves["pretrained encoder"][0] > curves["random encoder"][0]   # it is the pretraining, not the network
assert lowT["pretrained encoder"] > lowT["raw samples"]                # a second task, same encoder

## Part C · The pretraining cohort matters

Now pretrain on a cohort in which nobody has a wide QRS (width factor 0.8 to 1.2), and probe for the same task.
Khattak et al. (2026) found that a contrastive ECG model's downstream performance depended on the
demographics and health status of its pretraining cohort. Here the effect is built in: the encoder never had to
tell wide complexes apart.

In [ ]:
narrow = fm.patient_params(2000, np.random.default_rng(3), width_range=(0.8, 1.2))
enc_narrow, _ = fm.pretrain(narrow, std, steps=400, seed=2)
cohort = {"full cohort": [], "no wide QRS in pretraining": []}
for n in budgets:
    cohort["full cohort"].append(curves["pretrained encoder"][budgets.index(n)])
    cohort["no wide QRS in pretraining"].append(probe_auc(enc_narrow.features, n))
for name, c in cohort.items():
    print(f"{name:28s}", "  ".join(f"n={n}: {a:.3f}" for n, a in zip(budgets, c)))
print(f"{'raw samples':28s}", "  ".join(f"n={n}: {a:.3f}" for n, a in zip(budgets, curves['raw samples'])))

In [ ]:
# Checks
assert all(f > m for f, m in zip(cohort["full cohort"], cohort["no wide QRS in pretraining"]))

## Part D · Attention and the square root of d

Attention (eq-m43-attention) weights values by a softmax of query–key dot products. For query and key entries that
are independent with mean 0 and variance 1, the dot product has variance $d$ (eq-m43-scaling). Without the
division by $\sqrt d$, scores grow with $d$ and the softmax becomes nearly one-hot: one key takes almost all the
weight, and gradients through the others vanish.

In [ ]:
rows = []
for d in (4, 16, 64, 256):
    raw_var, scaled_var = fm.score_variance(d, n=20000, rng=np.random.default_rng(d))
    r = np.random.default_rng(1)
    Q, K, V = r.normal(size=(200, d)), r.normal(size=(32, d)), r.normal(size=(32, 1))
    _, A = fm.attention(Q, K, V)
    _, A_raw = fm.attention(Q * np.sqrt(d), K, V)      # undoes the scaling: softmax(Q K^T) V
    rows.append((d, raw_var, scaled_var, A.max(axis=1).mean(), A_raw.max(axis=1).mean()))
print("   d   Var(q.k)  Var(q.k/sqrt d)  mean top weight (scaled)  (unscaled)")
for d, rv, sv, a, ar in rows:
    print(f"{d:4d}  {rv:8.1f}  {sv:15.2f}  {a:24.2f}  {ar:10.2f}")

In [ ]:
# Checks
for d, rv, sv, a, ar in rows:
    assert abs(rv / d - 1) < 0.05 and abs(sv - 1) < 0.05
assert rows[-1][4] > 0.9 and rows[-1][3] < 0.5     # unscaled attention collapses onto one key at large d

## Part E · Retrieval over the course text

Retrieval-augmented generation (Lewis et al. 2020) puts passages from a chosen collection in front of a language
model before it answers. The retrieval step needs no language model at all. Here the collection is this course's
glossary and formula sheet, the method is TF-IDF with cosine similarity (eq-m43-tfidf, eq-m43-cosine), and the
test set is 24 plain-language questions written for the course, each with the glossary entries that answer it.

Be careful with the result: the same author wrote the questions and the newer glossary entries, which makes the
test easier than real questions from real users.

In [ ]:
glossary = rt.parse_glossary()
index = rt.Index(rt.course_corpus())
print(f"Corpus: {len(index.ids)} documents ({len(glossary)} glossary entries), vocabulary of {len(index.vocab)} words")

ranked, gold = [], []
for q, g in rt.QUESTIONS:
    hits = [i.split(":", 1)[1] for i, _ in index.search(q, k=len(index.ids)) if i.startswith("glossary:")]
    ranked.append(hits); gold.append(set(g))
ret = {f"recall@{k}": rt.recall_at_k(ranked, gold, k) for k in (1, 3, 5)}
ret["MRR"] = rt.mean_reciprocal_rank(ranked, gold)
print({k: round(v, 3) for k, v in ret.items()})
print("\nQuestions whose answer is not ranked first:")
for (q, g), r in zip(rt.QUESTIONS, ranked):
    rank = min(r.index(t) + 1 for t in g)
    if rank > 1:
        print(f"  rank {rank:2d}  {q}\n           top result: {r[0]}; answer: {' / '.join(g)}")

In [ ]:
# Checks
assert ret["recall@1"] <= ret["recall@3"] <= ret["recall@5"]
assert ret["recall@5"] > ret["recall@1"]        # some answers are found, but not first
missed = [q for (q, g), r in zip(rt.QUESTIONS, ranked) if min(r.index(t) for t in g) >= 5]
print("Not in the top 5:", missed)

Look at the misses. Lexical retrieval fails when the question and the entry use different words for the same
idea ("the patients at deployment differ" against "dataset shift"). Dense retrievers replace word counts with
embeddings learned by the same contrastive loss as part B, trained on matched text pairs; NV-Embed (Lee et al.,
NVIDIA) is one such text-embedding model. Whatever the retriever, measure recall on questions written by someone
else, before trusting what the model says on top of it.

## Part F · An evaluation harness for language-model answers

The harness asks every question three times and reports: accuracy on the first run with its 95% Wilson interval
(eq-m43-wilson); the share of questions answered identically in all three runs, and Cohen's kappa between runs 1
and 2 (m35); and the share of citations that resolve to an entry in `references.bib` or to a course module.

CI cannot call a real model, so the "models" are **scripted stand-ins** (`rt.ScriptedModel`):

- *closed book*: knows the answer key but gives a wrong term on 30% of calls, and cites one of three keys, one of
  which we invented. Both rates are ours, planted so we can check that the harness finds them.
- *retrieval*: answers with one of the top three retrieved glossary entries and cites its module; at temperature 0
  it always takes the top one.

So the numbers below test the harness, not any language model.

In [ ]:
keys = rt.bib_keys()
models = {
    "closed book (planted faults)": rt.ScriptedModel("closed_book", index, glossary, error_rate=0.3,
                                                      cited_keys=["vaswani2017", "oord2018cpc", "smith2021ecg"], seed=1),
    "retrieval, temperature 0": rt.ScriptedModel("retrieval", index, glossary, temperature=0.0, seed=1),
    "retrieval, temperature 0.2": rt.ScriptedModel("retrieval", index, glossary, temperature=0.2, seed=1),
}
results = {name: rt.evaluate(m, keys=keys) for name, m in models.items()}
for name, r in results.items():
    lo, hi = r["ci"]
    print(f"{name:30s} accuracy {r['accuracy']:.2f} (95% CI {lo:.2f}-{hi:.2f})  same in 3 runs {r['all_runs_agree']:.2f}  "
          f"kappa {r['kappa_runs_1_2']:.2f}  citations valid {r['citations_valid']:.2f}")

print("\nHow many independent questions for a 95% interval of +/- h around an accuracy near p?")
for p in (0.5, 0.8, 0.95):
    print(f"  p = {p}: " + ", ".join(f"h = {h}: {rt.questions_needed(p, h)}" for h in (0.10, 0.05, 0.02)))
lo, hi = rt.wilson_interval(28, 150)
print(f"\nExample: 28 correct of 150 ECGs (about 19%) gives a 95% Wilson interval of {lo:.1%} to {hi:.1%}")

In [ ]:
# Checks
cb, r0, r2 = (results[k] for k in models)
assert 0.5 < cb["citations_valid"] < 0.8          # one key in three was invented
assert cb["all_runs_agree"] < 1 and r0["all_runs_agree"] == 1.0
assert r2["kappa_runs_1_2"] < r0["kappa_runs_1_2"]
assert r0["citations_valid"] == 1.0
assert rt.questions_needed(0.5, 0.05) == 385

### Optional · your own model

To try a real model, set `EPML_RUN_REAL_LLM=1`, write `ask_real_model(prompt) -> str` for your provider, and wrap
it in a class with the same `answer(question, gold)` method as the stand-in. Use only course text: **never send
patient data to an external service.** CI skips this cell.

In [ ]:
if os.environ.get("EPML_RUN_REAL_LLM") == "1":
    def ask_real_model(prompt):
        raise NotImplementedError("Connect your own model here; send only course text.")

    class RealModel:
        def answer(self, question, gold=None):
            prompt = ("Answer with one term from the Open EP Mapping Lab glossary, then a citation key or module id.\n"
                      f"Question: {question}\nFormat: <term> | <citation>")
            term, _, cite = ask_real_model(prompt).partition("|")
            return {"answer": term.strip(), "citation": cite.strip()}

    print(rt.evaluate(RealModel(), keys=keys))
else:
    print("Skipped: set EPML_RUN_REAL_LLM=1 and fill in ask_real_model() to evaluate your own model.")

## Part G · Real ECGs: does pretraining on a few hundred unlabeled records help?

Pretraining set: lead I of up to 120 PTB-XL 1.0.3 records (Wagner et al. 2020) from fold 9, labels ignored.
Positive pairs are two beats from the same record. Probe set: up to 120 records from fold 10 labelled sinus
rhythm (SR) or AF (AFIB); folds keep each patient in one fold, so no patient is in both sets. The probe sees the
mean beat of a record, so it can use beat shape (for example a missing P wave) but not RR irregularity.

License: Creative Commons Attribution 4.0. Every file is checked against the published SHA-256 list. Without
internet access this part is skipped. The foundation models cited in m43 pretrain on 10^5 to 10^6 or more ECGs;
whatever happens here with 120 records is a lesson about scale, not about those models.

In [ ]:
from epml import aiecg as ai
from epml.data import DataUnavailable, load_ptbxl_lead

try:
    pre_sig, _, pre_ids, _ = load_ptbxl_lead(n_per_class=60, codes=("SR", "AFIB"), fold=9)
    sig, lab, ids, counts = load_ptbxl_lead(n_per_class=60, codes=("SR", "AFIB"), fold=10)
    REAL = True
except DataUnavailable as err:
    print("Part G skipped:", err)
    REAL = False

In [ ]:
if REAL:
    FS_P = 100.0
    def beats(x):
        return fm.beats_from_record(x, ai.detect_r_peaks(np.asarray(x, float), FS_P), FS_P)
    pre_beats = [b for b in (beats(x) for x in pre_sig) if len(b) >= 4]
    std_real = fm.Standardizer(np.vstack(pre_beats))

    def sample(r, b):
        recs = r.choice(len(pre_beats), size=b, replace=len(pre_beats) < b)
        pairs = [r.choice(len(pre_beats[i]), size=2, replace=False) for i in recs]
        X1 = np.array([pre_beats[i][p[0]] for i, p in zip(recs, pairs)])
        X2 = np.array([pre_beats[i][p[1]] for i, p in zip(recs, pairs)])
        return std_real(X1), std_real(X2)

    enc_real, loss_real = fm.pretrain_pairs(sample, d_in=pre_beats[0].shape[1], steps=300, batch=64, seed=2)
    probe_beats = [beats(x) for x in sig]
    keep = [i for i, b in enumerate(probe_beats) if len(b) >= 2]
    y_real = np.array([lab[i] == "AFIB" for i in keep], float)
    mean_raw = np.array([std_real(probe_beats[i]).mean(axis=0) for i in keep])
    mean_feat = np.array([enc_real.features(std_real(probe_beats[i])).mean(axis=0) for i in keep])
    print(f"Pretraining: {len(pre_beats)} records, {sum(len(b) for b in pre_beats)} beats; "
          f"loss {loss_real[:20].mean():.2f} -> {loss_real[-20:].mean():.2f} (chance {np.log(64):.2f})")
    print(f"Probe set: {len(keep)} records, {int(y_real.sum())} AF")

    aucs = {"raw mean beat": [], "pretrained features": []}
    r = np.random.default_rng(0)
    pos, neg = np.flatnonzero(y_real == 1), np.flatnonzero(y_real == 0)
    for _ in range(20):
        tr = np.r_[r.choice(pos, 5, replace=False), r.choice(neg, 5, replace=False)]
        te = np.setdiff1d(np.arange(len(keep)), tr)
        for name, F, lam in [("raw mean beat", mean_raw, 10.0), ("pretrained features", mean_feat, 1.0)]:
            w = fm.fit_logistic(F[tr], y_real[tr], lam=lam)
            aucs[name].append(auc_rank(fm.logistic_score(w, F[te]), y_real[te]))
    for name, a in aucs.items():
        print(f"{name:20s} AUC with 10 labelled records: mean {np.mean(a):.3f} (range {np.min(a):.2f}-{np.max(a):.2f}, 20 draws)")

In [ ]:
# Checks
if REAL:
    assert loss_real[-20:].mean() < loss_real[:20].mean()
    assert all(0 <= np.mean(a) <= 1 for a in aucs.values())

## Part H · Optional: contrastive pretraining with a 1-D CNN in PyTorch

The same synthetic patients and loss as part B, with a small convolutional encoder whose global average pooling
makes its features insensitive to small time shifts. The loss is written as a cross-entropy over the similarity
matrix, which is exactly eq-m43-infonce. Runs only if PyTorch is installed.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part H skipped.")

if torch is not None:
    torch.manual_seed(0)
    body = torch.nn.Sequential(
        torch.nn.Conv1d(1, 16, 9, stride=2, padding=4), torch.nn.ReLU(),
        torch.nn.Conv1d(16, 32, 9, stride=2, padding=4), torch.nn.ReLU(),
        torch.nn.AdaptiveAvgPool1d(1), torch.nn.Flatten())
    head = torch.nn.Linear(32, 16)
    opt = torch.optim.Adam(list(body.parameters()) + list(head.parameters()), lr=2e-3)
    r = np.random.default_rng(2)
    t_losses = []
    for step in range(300):
        idx = r.choice(2000, size=128, replace=False)
        x1 = torch.tensor(std(fm.recordings(pre_params, idx, r)), dtype=torch.float32).unsqueeze(1)
        x2 = torch.tensor(std(fm.recordings(pre_params, idx, r)), dtype=torch.float32).unsqueeze(1)
        z1 = torch.nn.functional.normalize(head(body(x1)), dim=1)
        z2 = torch.nn.functional.normalize(head(body(x2)), dim=1)
        logits = z1 @ z2.T / 0.1
        target = torch.arange(128)
        loss = 0.5 * (torch.nn.functional.cross_entropy(logits, target) + torch.nn.functional.cross_entropy(logits.T, target))
        opt.zero_grad(); loss.backward(); opt.step()
        t_losses.append(loss.item())

    def cnn_features(X):
        with torch.no_grad():
            return body(torch.tensor(X, dtype=torch.float32).unsqueeze(1)).numpy()
    cnn_auc = probe_auc(cnn_features, 10)
    print(f"CNN loss {np.mean(t_losses[:20]):.2f} -> {np.mean(t_losses[-20:]):.2f}; probe AUC with 10 labels {cnn_auc:.3f} "
          f"(NumPy encoder {curves['pretrained encoder'][0]:.3f}, raw samples {curves['raw samples'][0]:.3f})")

In [ ]:
# Checks
if torch is not None:
    assert np.mean(t_losses[-20:]) < 0.8 * np.log(128)
    assert cnn_auc > 0.5

## Where to go next

- In part B, change the temperature $\tau$ in `fm.pretrain` (0.05, 0.5). What happens to the loss and to the probe?
- In part C, pretrain on a cohort with *only* wide complexes. Which task suffers now?
- In part E, write five questions of your own in words that do not appear in the glossary. What is recall@5?
- In part F, how many questions would you need to tell an accuracy of 0.70 from 0.80 with confidence?